In [182]:
import numpy as np
import matplotlib.pyplot as plt

In [183]:
def initialize_params(f, num_channels, filters):
    
    W = np.random.randn(filters, f, f, num_channels) * np.sqrt(2 / f)
    b = np.random.randn(filters, 1, 1, 1) * np.sqrt(2 / f)

    return (W, b)

In [184]:
def zero_padding(X, padding):
  
    X_pad = np.pad(X, ((0, 0), (padding, padding), (padding, padding), (0, 0)), mode="constant", constant_values=(0, 0))    
    
    return X_pad

In [185]:
n = np.ones((1, 3, 3, 1))
o = zero_padding(n, 3)
print(o.shape)
print(n.shape)

(1, 9, 9, 1)
(1, 3, 3, 1)


In [186]:
def x_conv_shape(n_pre, f, s):
    
    n = int((n_pre - f / s)) + 1
    
    return n

In [187]:
n = 10
f = 3
p = 2
s =1
nn = x_conv_shape(n, f, s)
nn

8

In [188]:
def relu(Z):
    
    A = np.maximum(0, Z)
    cache = Z
    
    return A, cache

In [189]:
def conv_1_filter(X, W, nh, nw, stride):
    
    (f, f, n_c) = W.shape
    X_conv = np.zeros((nh, nw))

    for k in range(n_c):

        for i in range(nh):
            v_start = i * stride
            v_end =  v_start + f
            for j in range(nw):
                h_start = j * stride
                h_end =  h_start + f
                
                frame = X[v_start:v_end, h_start:h_end, k]
                X_conv[i, j] += np.sum(frame * W[:, :, k])        
            
    return X_conv

In [195]:
x = np.ones((10, 10, 2))
x_pad = np.ones((14, 14, 2))
w = np.random.randn(3, 3, 2)
x_conv = conv_1_filter(x,  w, 3, 3, 3)
r,c = relu(x_conv)
print(r.shape)
print(c.shape)
print(x.shape)
print(w.shape)
print(x_conv.shape)

(3, 3)
(3, 3)
(10, 10, 2)
(3, 3, 2)
(3, 3)


In [198]:
def conv2d(X, filters, kernel_size, padding, strides, parameters=None, layer=None):
    """
    Applies convolutional process
    Args:
        - X(ndarray): input data with the shape of (m, n_h, n_W, n_C)
        - filters(int): number of applied filters
        - kernel_size(tuple): size of the applied filters (f, f)
        - padding(int): number of padding columns or rows
        - strides(int): number of strides
        - parameters(dictionary): contains W and b matrices
        - layer(int): layer number
    """

    num_channels = X[0].shape[-1]
    f = kernel_size[0]
    

    if not parameters:
        (W, b) = initialize_params(f=f, num_channels=num_channels, filters=filters)
    else:
        (W, b) = parameters[f"W{layer}"], parameters[f"b{layer}"]
      
    linear_cache = (X, W, b, filters, kernel_size, padding, strides, layer)
    X_pad = zero_padding(X=X, padding=padding)
    
    nh = x_conv_shape(n_pre=X_pad[0].shape[0], f=f, s=strides)
    nw = x_conv_shape(n_pre=X_pad[0].shape[1], f=f, s=strides)
   
    m = X_pad.shape[0]

    X_conv = np.zeros((m, nh, nw, filters))
    activation_cache = np.zeros((m, nh, nw, filters))
    
    for i in range(m):
        X_temp = np.zeros((nh, nw, filters))
       
        for f in range(filters):
            
            Z = conv_1_filter(X=X_pad[i, :, :, :], W=W[f, :, :, :], nh=nh, nw=nw, stride=strides) + b[f]
            X_temp[:, :, f], activation_cache[i, :, :, f] = relu(Z=Z)
   
        X_conv[i, :, :, :] = X_temp

    caches = (linear_cache, activation_cache)

    return X_conv, caches      

In [201]:
x = np.ones((12, 6, 6, 2))
w = np.random.randn(12, 3, 3, 2)
b = np.random.randn(12, 1, 1, 1)
x_conv1, c1 = conv2d(X=x, filters=4, kernel_size=(3, 3), padding=2, strides=1, parameters=None, layer=None)
x_conv2, c2 = conv2d(X=x, filters=4, kernel_size=(3, 3), padding=2, strides=1, parameters={"W1": w, "b1": b}, layer=1)
print(x_conv1.shape)
print(x_conv2.shape)

(12, 8, 8, 4)
(12, 8, 8, 4)
